In [ ]:
# ==============================================================================
# INTEGRATED HYBRID WOA-QNN, STANDALONE QNN, & CATBOOST COMPARATIVE PIPELINE
# ==============================================================================

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
import pennylane as qml
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ------------------------------------------------------------------------------
# STEP 1: GLOBAL ENVIRONMENT CONFIGURATIONS & PLOT AESTHETICS
# ------------------------------------------------------------------------------
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 11
sns.set_style("ticks")

# Enforce deterministic initialization across numpy operations
np.random.seed(42)

# ------------------------------------------------------------------------------
# STEP 2: DATA INGESTION & BASIN-WIDE ROBUST SCALING
# ------------------------------------------------------------------------------
print("--- Step 1: Data Ingestion and Basin-Wide Preprocessing ---")

# Load real independent multi-well CSV datasets
train_df = pd.read_csv('well1_csv') 
test_df = pd.read_csv('well2.csv')

# Explicitly define the target 6-dimensional petrophysical wireline suite
feature_names = ['GR', 'SP', 'DT', 'NPHI', 'RHOB', 'LLD']

# Execute listwise deletion to remove any NaN entries and preserve coherence
train_df = train_df.dropna(subset=feature_names + ['TOC'])
test_df = test_df.dropna(subset=feature_names + ['TOC'])

X_train_raw = train_df[feature_names].values
y_train = train_df['TOC'].values  

X_test_raw = test_df[feature_names].values
y_test = test_df['TOC'].values    

print(f" -> Well 1 (Calibration/Training Profiles) Row Count: {X_train_raw.shape[0]}")
print(f" -> Well 2 (Blind Basin Testing Profiles) Row Count: {X_test_raw.shape[0]}")

# Record the global absolute limits for linear boundary translation
min_toc_val = min(y_train.min(), y_test.min())
max_toc_val = max(y_train.max(), y_test.max())

# Apply joint global feature scaling to prevent quantum state over-rotation
global_X = np.vstack([X_train_raw, X_test_raw])
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_X.fit(global_X)

X_train = scaler_X.transform(X_train_raw)
X_test = scaler_X.transform(X_test_raw)
print(" -> Basin-wide joint MinMax feature transformation completed.")

# ------------------------------------------------------------------------------
# STEP 3: CLASSICAL BENCHMARK MODEL (CATBOOST REGRESSOR)
# ------------------------------------------------------------------------------
print("\n--- Step 2: Evaluating Classical Baseline (CatBoost) ---")

# Initialize GBDT using Permutation-Driven Ordered Boosting and Oblivious Trees
catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function='RMSE',
    verbose=0,
    random_seed=42
)

# Train baseline using Well 1 continuous array profiles
start_time = time.time()
catboost_model.fit(X_train, y_train)
cat_train_time = time.time() - start_time

# Predict blindly over the withheld Well 2 space
cat_preds = catboost_model.predict(X_test)

# Compute performance parameters
cat_rmse = np.sqrt(mean_squared_error(y_test, cat_preds))
cat_mae = mean_absolute_error(y_test, cat_preds)
cat_r2 = r2_score(y_test, cat_preds)

print(f" -> CatBoost Computational Training Runtime: {cat_train_time:.2f} seconds")
print(f" -> CatBoost Blind Test Coefficient of Determination (R²)  : {cat_r2:.4f}")

# ------------------------------------------------------------------------------
# STEP 4: QUANTUM MACHINE LEARNING STRUCTURAL FOUNDATION (QNN)
# ------------------------------------------------------------------------------
print("\n--- Step 3: Initializing Variational Quantum Circuit Topology ---")

num_qubits = len(feature_names)  # 6 Qubits representing 6 logs
dev = qml.device("default.qubit", wires=num_qubits)

@qml.qnode(dev)
def quantum_neural_network(weights, features):
    """
    Topological blueprint of the 6-qubit Parameterized Quantum Circuit (PQC).
    """
    # 1. Quantum State Preparation via Angle Encoding
    for i in range(num_qubits):
        clipped_feature = np.clip(features[i], 0.0, 1.0)
        qml.RY(clipped_feature * np.pi, wires=i)  
        
    # 2. Hidden Processing Layers (Hardware-Efficient Ansatz)
    num_layers = weights.shape[0]
    for layer in range(num_layers):
        for i in range(num_qubits):
            qml.RY(weights[layer, i], wires=i)
        
        # Symmetrical circular CNOT entanglement cascade
        for i in range(num_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[num_qubits - 1, 0])  

    # 3. Collective multi-qubit joint state measurement
    return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]

def forward_pass(weights, X_data, min_toc, max_toc):
    """Computes expectations and maps native bounds back to target geological space."""
    predictions = []
    for sample in X_data:
        raw_quantum_outputs = quantum_neural_network(weights, sample)
        avg_quantum_output = np.mean(raw_quantum_outputs)
        
        # Real-world scale transformation layer bypassing single-qubit saturation limits
        scaled_output = min_toc + (avg_quantum_output + 1.0) * 0.5 * (max_toc - min_toc)
        predictions.append(scaled_output)
    return np.array(predictions)

# ------------------------------------------------------------------------------
# STEP 5: STANDALONE UNOPTIMIZED QNN BENCHMARK EVALUATION
# ------------------------------------------------------------------------------
print("\n--- Step 4: Evaluating Standalone Unoptimized QNN Baseline ---")

quantum_layers = 2
weight_dimensions = (quantum_layers, num_qubits)  # Structural shape (2, 6)

# Generate unoptimized, standard random initialization weights
unoptimized_weights = np.random.uniform(-0.5, 0.5, weight_dimensions)

# Generate blind predictions directly to quantify the barren plateau effect
qnn_base_preds = forward_pass(unoptimized_weights, X_test, min_toc_val, max_toc_val)

qnn_base_rmse = np.sqrt(mean_squared_error(y_test, qnn_base_preds))
qnn_base_mae = mean_absolute_error(y_test, qnn_base_preds)
qnn_base_r2 = r2_score(y_test, qnn_base_preds)

print(f" -> Standalone QNN Blind Test Coefficient of Determination (R²): {qnn_base_r2:.4f}")

# ------------------------------------------------------------------------------
# STEP 6: METAHEURISTIC SWARM ARCHITECTURE (WHALE OPTIMIZATION)
# ------------------------------------------------------------------------------
print("\n--- Step 5: Loading Global Whale Optimisation Mechanics ---")

def objective_function(weights_flat, X, y, shape, min_toc, max_toc):
    weights = weights_flat.reshape(shape)
    predictions = forward_pass(weights, X, min_toc, max_toc)
    return mean_squared_error(y, predictions)

def whale_optimization_algorithm(X, y, num_whales, max_iter, weight_shape, min_toc, max_toc):
    dim = np.prod(weight_shape)
    
    # Initialize whale search paths within stable, narrow local operational bounds [-0.5, 0.5]
    whale_positions = np.random.uniform(-0.5, 0.5, (num_whales, dim))
    best_score = float("inf")
    best_whale = np.zeros(dim)
    
    # Evaluate baseline swarm parameters
    for i in range(num_whales):
        score = objective_function(whale_positions[i], X, y, weight_shape, min_toc, max_toc)
        if score < best_score:
            best_score = score
            best_whale = whale_positions[i].copy()

    # Dynamic Optimization Loops bypassing Barren Plateaus
    for t in range(max_iter):
        a = 2 - t * (2 / max_iter)  # Linear decelerator decay
        for i in range(num_whales):
            r1, r2, p, l = np.random.rand(), np.random.rand(), np.random.rand(), np.random.uniform(-1, 1)
            A = 2 * a * r1 - a
            C = 2 * r2
            b = 1  # Helical shape indicator constant
            
            if p < 0.5:
                if abs(A) < 1:
                    # Encircling Prey (Exploitation Phase)
                    D = abs(C * best_whale - whale_positions[i])
                    whale_positions[i] = best_whale - A * D
                else:
                    # Search for Prey (Global Exploration Phase)
                    random_whale = whale_positions[np.random.randint(0, num_whales)]
                    D = abs(C * random_whale - whale_positions[i])
                    whale_positions[i] = random_whale - A * D
            else:
                # Bubble-net Spiral Attack Matrix
                distance_to_leader = abs(best_whale - whale_positions[i])
                whale_positions[i] = distance_to_leader * np.exp(b * l) * np.cos(2 * np.pi * l) + best_whale
            
            # Clip weight limits to prevent extreme chaotic parameter over-rotations
            whale_positions[i] = np.clip(whale_positions[i], -1.0, 1.0)
            
            # Evaluate objective fitness metrics
            score = objective_function(whale_positions[i], X, y, weight_shape, min_toc, max_toc)
            if score < best_score:
                best_score = score
                best_whale = whale_positions[i].copy()
                
        if (t + 1) % 5 == 0 or t == 0:
            print(f" -> Iteration {t+1}/{max_iter} - Best Global Swarm Training RMSE: {np.sqrt(best_score):.4f} wt.%")
        
    return best_whale.reshape(weight_shape)

# ------------------------------------------------------------------------------
# STEP 7: MODEL EXECUTION & HYBRID MODEL TESTING
# ------------------------------------------------------------------------------
print("\n--- Step 6: Commencing WOA-QNN Quantum Training Matrix ---")

# Operational swarm dimensions
whales_count = 25       
iteration_loops = 35   

# Calibrate parameters exclusively on Well 1 profiles
start_q_time = time.time()
optimized_weights = whale_optimization_algorithm(
    X_train, y_train, num_whales=whales_count, max_iter=iteration_loops, 
    weight_shape=weight_dimensions, min_toc=min_toc_val, max_toc=max_toc_val
)
qnn_train_time = time.time() - start_q_time

# Execute a blind cross-validation sweep over the unseen testing dataset (Well 2)
qnn_hybrid_preds = forward_pass(optimized_weights, X_test, min_toc_val, max_toc_val)

# Extract testing evaluation indicators
qnn_hybrid_rmse = np.sqrt(mean_squared_error(y_test, qnn_hybrid_preds))
qnn_hybrid_mae = mean_absolute_error(y_test, qnn_hybrid_preds)
qnn_hybrid_r2 = r2_score(y_test, qnn_hybrid_preds)

print("\n==================================================================")
print("             HYBRID WOA-QNN GEOSCIENCE METRICS GENERATED          ")
print("==================================================================")
print(f" -> WOA-QNN Computational Training Time: {qnn_train_time:.2f} seconds")
print(f" -> Evaluation Target Profile: well2.csv (Blind Basin Testing Well)")
print(f" -> Testing Coefficient of Determination (R²): {qnn_hybrid_r2:.4f}")

# ------------------------------------------------------------------------------
# STEP 8: GRAPHICS GENERATION & EVALUATION
# ------------------------------------------------------------------------------
print("\n--- Step 7: Rendering Publication Quality Graphics ---")

# Display complete three-way model results summary table
summary_df = pd.DataFrame({
    'Performance Metric': ['R²', 'RMSE (wt.%)', 'MAE (wt.%)'],
    'Classical CatBoost Baseline': [cat_r2, cat_rmse, cat_mae],
    'Standalone Standard QNN': [qnn_base_r2, qnn_base_rmse, qnn_base_mae],
    'Proposed Hybrid WOA-QNN': [qnn_hybrid_r2, qnn_hybrid_rmse, qnn_hybrid_mae]
})
print("\nComparative Model Statistics Table Summary:")
print(summary_df.to_string(index=False))

# Build three-panel 1:1 Geochemical Predicted vs Measured cross-plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot Panel 1: CatBoost
axes[0].scatter(y_test, cat_preds, color='#d95f02', alpha=0.6, edgecolors='k', s=45, label='CatBoost')
axes[0].plot([min_toc_val-0.5, max_toc_val+0.5], [min_toc_val-0.5, max_toc_val+0.5], 'r--', linewidth=2, label='1:1 Line')
axes[0].set_title(f'Classical CatBoost Baseline\n(Blind Test $R^2$: {cat_r2:.3f})', fontweight='bold', fontsize=11)
axes[0].set_xlabel('Core Measured TOC (wt.%)', labelpad=8)
axes[0].set_ylabel('Model Predicted TOC (wt.%)', labelpad=8)
axes[0].set_xlim(min_toc_val-0.3, max_toc_val+0.3)
axes[0].set_ylim(min_toc_val-0.3, max_toc_val+0.3)
axes[0].grid(True, linestyle=':', alpha=0.5)
axes[0].legend(loc='upper left')

# Plot Panel 2: Standalone QNN Baseline
axes[1].scatter(y_test, qnn_base_preds, color='#7570b3', alpha=0.6, edgecolors='k', s=45, label='Standalone QNN')
axes[1].plot([min_toc_val-0.5, max_toc_val+0.5], [min_toc_val-0.5, max_toc_val+0.5], 'r--', linewidth=2, label='1:1 Line')
axes[1].set_title(f'Standalone Standard QNN\n(Blind Test $R^2$: {qnn_base_r2:.3f})', fontweight='bold', fontsize=11)
axes[1].set_xlabel('Core Measured TOC (wt.%)', labelpad=8)
axes[1].set_xlim(min_toc_val-0.3, max_toc_val+0.3)
axes[1].set_ylim(min_toc_val-0.3, max_toc_val+0.3)
axes[1].grid(True, linestyle=':', alpha=0.5)
axes[1].legend(loc='upper left')

# Plot Panel 3: Hybrid WOA-QNN
axes[2].scatter(y_test, qnn_hybrid_preds, color='#1f77b4', alpha=0.6, edgecolors='k', s=45, label='Hybrid WOA-QNN')
axes[2].plot([min_toc_val-0.5, max_toc_val+0.5], [min_toc_val-0.5, max_toc_val+0.5], 'r--', linewidth=2, label='1:1 Line')
axes[2].set_title(f'Proposed Hybrid WOA-QNN\n(Blind Test $R^2$: {qnn_hybrid_r2:.3f})', fontweight='bold', fontsize=11)
axes[2].set_xlabel('Core Measured TOC (wt.%)', labelpad=8)
axes[2].set_xlim(min_toc_val-0.3, max_toc_val+0.3)
axes[2].set_ylim(min_toc_val-0.3, max_toc_val+0.3)
axes[2].grid(True, linestyle=':', alpha=0.5)
axes[2].legend(loc='upper left')

plt.tight_layout()

# Save as a high-density vector layout file for journal submission
plt.savefig('TOC_Three_Way_Model_Comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(" -> Analysis complete. Comprehensive performance figure exported as 'TOC_Three_Way_Model_Comparison.png'.")